## 测试效果

- 测试代码: [speed_test.ipynb](speed_test.ipynb)
- 测试环境: Intel i5-12400 CPU, 48GB RAM, 1x NVIDIA GeForce RTX 4070
- 运行环境: Ubuntu 24.04.1 LTS, cuda 12.4, python 3.10.16
- 测试说明: 单任务执行的数据（非并发测试）


##### 使用**注意事项**，需要将该文件移动到 CosyVoise 目录下，并安装 Ipython 模块运行

## 默认情况下

In [ ]:
import time
import asyncio
import torchaudio

import sys
sys.path.append('third_party/Matcha-TTS')

from cosyvoice.cli.cosyvoice import  CosyVoice2
from cosyvoice.utils.file_utils import load_wav

prompt_text = '希望你以后能够做得比我还好哟'
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)

# cosyvoice = CosyVoice2('./pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=True)
cosyvoice = CosyVoice2('./pretrained_models/CosyVoice2-0.5B', load_jit=True, load_trt=False, fp16=True)

In [ ]:
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', prompt_text, prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', prompt_text, prompt_speech_16k, stream=True)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_zero_shot(text_generator(), prompt_text, prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_zero_shot(text_generator(), prompt_text, prompt_speech_16k, stream=True)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

In [ ]:
# instruct usage
for i, j in enumerate(cosyvoice.inference_instruct2('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '用四川话说这句话', prompt_speech_16k, stream=False)):
    torchaudio.save('1_instruct2_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


In [ ]:
# instruct usage
def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'
for i, j in enumerate(cosyvoice.inference_instruct2(text_generator(), '用四川话说这句话', prompt_speech_16k, stream=False)):
    torchaudio.save('instruct2_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


## 使用vllm加速llm推理


#### **注意：**

在 jupyter notebook 中，如果要运行下列代码，需要将vllm_use_cosyvoice2_model.py正确复制到 vllm 包中，并注册到 _VLLM_MODELS 字典中。

运行下面的 code 完成

In [1]:
import os
import shutil

# 获取vllm包的安装路径
try:
    import vllm
except ImportError:
    raise ImportError("vllm package not installed")


vllm_path = os.path.dirname(vllm.__file__)
print(f"vllm package path: {vllm_path}")

# 定义目标路径
target_dir = os.path.join(vllm_path, "model_executor", "models")
target_file = os.path.join(target_dir, "cosyvoice2.py")

# 复制模型文件
source_file = "./async_cosyvoice/vllm_use_cosyvoice2_model.py"
if not os.path.exists(source_file):
    raise FileNotFoundError(f"Source file {source_file} not found")

shutil.copy(source_file, target_file)
print(f"Copied {source_file} to {target_file}")

# 修改registry.py文件
registry_path = os.path.join(target_dir, "registry.py")
new_entry = '    "CosyVoice2Model": ("cosyvoice2", "CosyVoice2Model"),  # noqa: E501\n'

# 读取并修改文件内容
with open(registry_path, "r") as f:
    lines = f.readlines()

# 检查是否已存在条目
entry_exists = any("CosyVoice2Model" in line for line in lines)

if not entry_exists:
    # 寻找插入位置
    insert_pos = None
    for i, line in enumerate(lines):
        if line.strip().startswith("**_FALLBACK_MODEL"):
            insert_pos = i + 1
            break
    
    if insert_pos is None:
        raise ValueError("Could not find insertion point in registry.py")
    
    # 插入新条目
    lines.insert(insert_pos, new_entry)
    
    # 写回文件
    with open(registry_path, "w") as f:
        f.writelines(lines)
    print("Successfully updated registry.py")
else:
    print("Entry already exists in registry.py, skipping modification")

print("All operations completed successfully!")

/root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-05 13:52:22,246	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


vllm package path: /root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/vllm
Copied ./async_cosyvoice/vllm_use_cosyvoice2_model.py to /root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/vllm/model_executor/models/cosyvoice2.py
Entry already exists in registry.py, skipping modification
All operations completed successfully!


In [2]:
import time
import asyncio
import torch
import torchaudio

import sys
sys.path.append('third_party/Matcha-TTS')

from async_cosyvoice.async_cosyvoice import AsyncCosyVoice2
from cosyvoice.utils.file_utils import load_wav

prompt_text = '希望你以后能够做得比我还好哟'
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)

# cosyvoice = AsyncCosyVoice2('./pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=True)
cosyvoice = AsyncCosyVoice2('./pretrained_models/CosyVoice2-0.5B', load_jit=True, load_trt=False, fp16=True)

failed to import ttsfrd, use WeTextProcessing instead
[2025-05-05 13:52:28,587] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


2025-05-05 13:52:28,700 INFO gcc -pthread -B /root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /root/autodl-tmp/miniconda3/envs/cosyvoice2/include -fPIC -O2 -isystem /root/autodl-tmp/miniconda3/envs/cosyvoice2/include -fPIC -c /tmp/tmpwtq5tcem/test.c -o /tmp/tmpwtq5tcem/test.o
2025-05-05 13:52:28,724 INFO gcc -pthread -B /root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat /tmp/tmpwtq5tcem/test.o -laio -o /tmp/tmpwtq5tcem/a.out
/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
2025-05-05 13:52:29,406 INFO gcc -pthread -B /root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat -Wno-unused-result -Wsign-compare -DNDEBUG -fwrapv -O2 -Wall -fPIC -O2 -isystem /root/autodl-tmp/miniconda3/envs/cosyvoice2/include -fPIC -O2 -isystem /root/autodl-tmp/miniconda3/envs/cosyvoice2/include -fPIC -c /t

INFO 05-05 13:52:31 __init__.py:207] Automatically detected platform cuda.
WARNING 05-05 13:52:31 registry.py:352] Model architecture CosyVoice2Model is already registered, and will be overwritten by the new model class <class 'async_cosyvoice.vllm_use_cosyvoice2_model.CosyVoice2Model'>.


2025-05-05 13:52:31,985 - modelscope - INFO - PyTorch version 2.5.1 Found.
2025-05-05 13:52:31,987 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2025-05-05 13:52:32,033 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 bc8eec8039e6658eb28ac236cfd72410 and a total number of 980 components indexed
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-05-05 13:52:35,326 INFO input frame rate=25
/root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:14

INFO 05-05 13:53:16 config.py:2444] Downcasting torch.float32 to torch.float16.
INFO 05-05 13:53:16 config.py:549] This model supports multiple tasks: {'classify', 'embed', 'generate', 'score', 'reward'}. Defaulting to 'generate'.
INFO 05-05 13:53:16 config.py:1555] Chunked prefill is enabled with max_num_batched_tokens=1024.
INFO 05-05 13:53:16 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='./pretrained_models/CosyVoice2-0.5B', speculative_config=None, tokenizer='./pretrained_models/CosyVoice2-0.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observabilit

/root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/torch/utils/_device.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.01it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.01it/s]



INFO 05-05 13:53:18 gpu_model_runner.py:1060] Loading model weights took 0.9530 GB


NotImplementedError: 

: 

In [ ]:
i = 0
async for j in cosyvoice.inference_sft('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', spk_id='xiaohe', stream=False):
    torchaudio.save('sft_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    i += 1


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
task_id = 0

# text_generator 字符数较多的时，（bfloat16下）llm的稳定性降低，语音会错乱，建议手动切分，直接传入少量文本流式推理

def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'

tts_text = text_generator()

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
task_id = 0

def text_generator():
    yield '收到好友从远方寄来的生日礼物，'
    yield '那份意外的惊喜与深深的祝福'
    yield '让我心中充满了甜蜜的快乐，'
    yield '笑容如花儿般绽放。'

tts_text = text_generator()

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot(tts_text, prompt_text, prompt_speech_16k, stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
# instruct usage
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物[breath]，那份意外的惊喜与深深的祝福[breath]让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2(tts_text, '用四川话说这句话', prompt_speech_16k, stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')

In [14]:
import random
from funasr import AutoModel
from cosyvoice.utils.file_utils import load_wav


ser_model_name = "emotion2vec_base_finetuned"
prompt_speech_16k = load_wav('./asset/zero_shot_prompt.wav', 16000)
ser_model = AutoModel(model=f"iic/{ser_model_name}")
ser_res = ser_model.generate(prompt_speech_16k, granularity="utterance", extract_embedding=True)

funasr version: 1.2.6.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel


2025-05-05 15:59:52,160 - modelscope - INFO - PyTorch version 2.5.1 Found.
2025-05-05 15:59:52,163 - modelscope - INFO - Loading ast index from /root/.cache/modelscope/ast_indexer
2025-05-05 15:59:52,210 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 bc8eec8039e6658eb28ac236cfd72410 and a total number of 980 components indexed


You are using the latest version of funasr-1.2.6


/root/autodl-tmp/miniconda3/envs/cosyvoice2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-05-05 15:59:53,372] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/root/autodl-tmp/miniconda3/envs/cosyvoice2/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/root

In [ ]:
# instruct 不能使用 Generater 模式传入text
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物[breath]，那份意外的惊喜与深深的祝福[breath]让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2(tts_text, '用四川话说这句话', prompt_speech_16k, stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')

In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot_by_spk_id(tts_text, spk_id='xiaohe', stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_zero_shot_by_spk_id(tts_text, spk_id='xiaohe', stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('zero_shot_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2_by_spk_id(tts_text, '使用四川话说', spk_id='xiaohe', stream=False):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
task_id = 0

tts_text = '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'

chunk_num = 0
audio_data: torch.Tensor = None
async for chunk in cosyvoice.inference_instruct2_by_spk_id(tts_text, '使用四川话说', spk_id='xiaohe', stream=True):
    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
    chunk_num += 1
torchaudio.save('instruct2_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
print(f'任务完成，生成 {chunk_num} 个片段')


In [ ]:
import time
import asyncio
import torchaudio
from typing import AsyncGenerator

async def test_concurrent_instruct(num_tasks: int = 3, semaphore_limit: int = 5, stream=False):
    """
    异步并发测试函数，用于验证多个推理任务并行执行能力
    
    参数：
    num_tasks: 并发任务数量，默认3个
    
    功能特点：
    1. 动态创建多个异步生成器任务
    2. 使用信号量控制并发度（默认限制5个）
    3. 实时跟踪任务完成进度
    4. 自动处理异常并记录错误
    """
    semaphore = asyncio.Semaphore(semaphore_limit)  # 并发控制
    
    async def single_task(task_id: int):
        """单个推理任务处理流程"""
        async with semaphore:
            try:
                # text_gen = (chunk for chunk in [
                #     f'这是任务{task_id}的第一句话，',
                #     f'测试并发处理能力，',
                #     f'当前进度：{task_id}-第三部分'
                # ])
                text_gen = f'''这是任务{task_id}，收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。'''
                # 记录保存索引
                save_index = 0
                
                # 流式处理
                audio_data: torch.Tensor = None
                async for chunk in cosyvoice.inference_zero_shot_by_spk_id(
                    text_gen,
                    'xiaohe',
                    # prompt_text,
                    # prompt_speech_16k,
                    stream=stream,
                ):
                    audio_data = torch.concat([audio_data, chunk['tts_speech']], dim=1) if audio_data is not None else chunk['tts_speech']
                    save_index += 1
                    print(f'任务 {task_id} 进度：{save_index}')
                # 保存音频片段
                torchaudio.save('tts_speech_{}.wav'.format(task_id), audio_data, cosyvoice.sample_rate)
                    
                print(f'任务 {task_id} 完成，生成 {save_index} 个片段')
                
            except Exception as e:
                print(f'任务 {task_id} 异常: {str(e)}')
    
    # 创建并发任务
    tasks = [single_task(i) for i in range(num_tasks)]
    
    # 执行并等待
    await asyncio.gather(*tasks)


In [ ]:
start_time = time.time()
await test_concurrent_instruct(5, semaphore_limit=10)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(10, semaphore_limit=10)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(20, semaphore_limit=20)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(25, semaphore_limit=25)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(2, semaphore_limit=5, stream=True)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(5, semaphore_limit=5, stream=True)
print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
start_time = time.time()
await test_concurrent_instruct(10, semaphore_limit=10, stream=True)
print("--- %s seconds ---" % (time.time() - start_time))

## 新增 spk_info 方法

In [ ]:
from cosyvoice.utils.file_utils import load_wav

prompt_text = '希望你以后能够做的比我还好呦'
prompt_speech_16k = load_wav('/home/qihua/音乐/希望你以后能够做的比我还好呦.wav', 16000)
cosyvoice.frontend.generate_spk_info(
    '001',
    prompt_text=prompt_text,
    prompt_speech_16k=prompt_speech_16k,
    name='系统默认'
)

In [ ]:
from cosyvoice.utils.file_utils import load_wav

prompt_text = '以温润磁性的声线，宛如夏日细雨。'
prompt_speech_16k = load_wav('/home/qihua/音乐/（龙小夏）以温润磁性的声线，宛如夏日细雨.wav', 16000)
cosyvoice.frontend.generate_spk_info(
    'longxiaoxia',
    prompt_text=prompt_text,
    prompt_speech_16k=prompt_speech_16k,
    name='龙小夏'
)

In [ ]:
from cosyvoice.utils.file_utils import load_wav

prompt_text = '今天天气真是太好了，阳光灿烂心情超级棒'
prompt_speech_16k = load_wav('/home/qihua/音乐/(湾湾小何)今天天气真是太好了，阳光灿烂心情超级棒.wav', 16000)
cosyvoice.frontend.generate_spk_info(
    'xiaohe',
    prompt_text=prompt_text,
    prompt_speech_16k=prompt_speech_16k,
    name='湾湾小何'
)